# Librería SDV 
Autor: Emmanuel Peña Ruiz*
Fecha: 04 de Junio

In [1]:
#Importamos librerias 
import pandas as pd 
import numpy as np 
import random

In [2]:
import sdv #Generar datos sintéticos  
import sdmetrics #Evaluar la calidad de los datos generados

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

In [3]:
from sdmetrics.reports.single_table import QualityReport

In [4]:
print ("SDV", sdv.__version__)

SDV 1.37.0


In [5]:
#Creamos un Dataset (original) que servirá como fuente para crear los Datos sintéticos
dfClientes = pd.DataFrame(
    {
        "cliente_id": [1,2,3,4,5,6,7,8,9,10],
        "edad": [22,23,43,28,53,56,43,56,65,40],
        "ingreso_manual": [25000,15000,20000,10000,5000,17000,3000,12000,35000,7500],
        "ciudad": ["Veracruz", "Cordoba", "Paso del Macho", "Amatlan", "Fortin", "Cuitlahuac", "Yanga", "Cordoba", "Orizaba", "Cuitlahuac"]
    }
)

In [6]:
dfClientes.head()

,cliente_id,edad,ingreso_manual,ciudad
0,1,22,25000,Veracruz
1,2,23,15000,Cordoba
2,3,43,20000,Paso del Macho
3,4,28,10000,Amatlan
4,5,53,5000,Fortin


In [7]:
#Definir los los metadatos 
metadata = SingleTableMetadata()

In [8]:
metadata.detect_from_dataframe(
    data = dfClientes
)

In [9]:
metadata.visualize()

C:\Users\penar\anaconda3\Lib\site-packages\sdv\metadata\visualization.py:131: RuntimeWarning: Graphviz does not seem to be installed on this system. For full metadata visualization capabilities, please make sure to have its binaries propertly installed: https://graphviz.gitlab.io/download/
  warnings.warn(warning_message, RuntimeWarning)


ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [10]:
#Guardamos el metadata en un archivo JSON 
metadata.save_to_json("dfClientes_metadata.json")

In [11]:
#Entrenamos el modelo 
synthesizer = GaussianCopulaSynthesizer (
    metadata
)

C:\Users\penar\anaconda3\Lib\site-packages\sdv\single_table\base.py:182: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)


In [13]:
#Entrenamiento 
synthesizer.fit(
    dfClientes
)

In [15]:
#Generamos datos sintéticos 
clientes_sinteticos = synthesizer.sample(
    num_rows=100
)

In [16]:
clientes_sinteticos.head()

,cliente_id,edad,ingreso_manual,ciudad
0,16169768,58,24738,Yanga
1,4918803,45,7290,Fortin
2,1900081,45,18118,Amatlan
3,531516,45,10579,Cordoba
4,10211768,59,12529,Cuitlahuac


In [17]:
clientes_sinteticos.describe(include="all")

,cliente_id,edad,ingreso_manual,ciudad
count,1.000000e+02,100.000000,100.000000,100
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Cuitlahuac
freq,NaN,NaN,NaN,22
mean,8.532170e+06,42.940000,15994.410000,NaN
std,4.687773e+06,13.119359,8301.337621,NaN
min,4.660500e+04,22.000000,5006.000000,NaN
25%,5.008268e+06,30.000000,8653.500000,NaN
50%,8.656048e+06,45.000000,14916.000000,NaN
75%,1.256688e+07,54.250000,22312.000000,NaN


In [18]:
#Dataset contaminado 
dfClientesGIGO = clientes_sinteticos.copy()

In [19]:
#Colocar edades imposibles 
indices = random.sample(list(dfClientesGIGO.index),5)


In [20]:
dfClientesGIGO.loc[indices, "edad"] = -5

In [21]:
#Introducir valores nulos 
duplicados = dfClientesGIGO.sample(10,random_state=42)

In [22]:
dfClientesGIGO = pd.concat([dfClientesGIGO, duplicados], ignore_index=True)

In [23]:
#Verificamos valores nulos
dfClientesGIGO.duplicated().sum()

np.int64(10)

In [24]:
dfClientesGIGO.describe(include="all")

,cliente_id,edad,ingreso_manual,ciudad
count,1.100000e+02,110.000000,110.000000,110
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,23
mean,8.614045e+06,39.690909,16057.790909,NaN
std,4.775996e+06,16.333534,8211.337287,NaN
min,4.660500e+04,-5.000000,5006.000000,NaN
25%,4.948624e+06,27.000000,8861.750000,NaN
50%,8.956428e+06,41.500000,14916.000000,NaN
75%,1.274153e+07,53.750000,22246.500000,NaN


In [25]:
#Generamos el reporte 
reporte = QualityReport()

In [27]:
reporte.generate (
    real_data = dfClientes, 
    synthetic_data =clientes_sinteticos, 
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████████████████████████████████████████████████| 4/4 [00:00<00:00, 599.87it/s]|
Column Shapes Score: 86.0%

(2/2) Evaluating Column Pair Trends: |█████████████████████████████████████████████████| 6/6 [00:00<00:00, 167.90it/s]|
Column Pair Trends Score: 25.0%

Overall Score (Average): 55.5%



In [28]:
reporte.generate (
    real_data = dfClientes, 
    synthetic_data = dfClientesGIGO, 
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████████████████████████████████████████████████| 4/4 [00:00<00:00, 850.69it/s]|
Column Shapes Score: 85.45%

(2/2) Evaluating Column Pair Trends: |█████████████████████████████████████████████████| 6/6 [00:00<00:00, 299.04it/s]|
Column Pair Trends Score: 16.82%

Overall Score (Average): 51.14%

